In [ ]:
from backtest_engine import *
import numpy as np
from datetime import date
import tomllib

with open("config.toml", "rb") as f:
    config = tomllib.load(f)["alpaca-api"]

df = request(
    ticker="SPY",
    config=config,
)

In [ ]:
# instantiate a portfolio backtest - behaviour is swapped via strategy=/execution=/cost_model= kwargs instead of subclassing
p = Portfolio(df, cost_model=DynamicCostModel())

print(p)
print(p.sharpe)

In [ ]:
# backtest result over the entire data range provided
# returns a Tearsheet object when decompose param is false (default)
t = p.report()

In [ ]:
# backtest decomposed into long and short legs
# returns a PortfolioDecomposer object
d = p.report(decompose=True)

In [ ]:
# backtest result over the provided date range
_ = p.report(start=date(2020, 1, 1), end=date(2021, 1, 1))

In [ ]:
# strategy actions over a specific day
# heavily tailored to visualise the SFI paper's strategy and will need to be rewritten for different strategies
# returns a DailySnapshot object
d = p.report(day=date(2025, 4, 12))

In [ ]:
# classmethod, visualises capacity of a strategy
Portfolio.sharpe_curve(df=df, max_aum=1_000_000_000, resolution=19, cost_model=DynamicCostModel(), execution_model=CappedVolumeRolloverExecution())
Portfolio.sharpe_curve(df=df, max_aum=1_000_000_000, resolution=19, cost_model=DynamicCostModel(), execution_model=CappedVolumeExecution())

In [ ]:
ps = Portfolio(df, cost_model=DynamicCostModel(), long_permissions=False) # margin-funded short hedge

toy_book, _ = generate_toy_equity( # generate a toy book equity series
    portfolio=ps,
    sharpe=1.5,
    volatility=0.18,
    beta=0.75,
    benchmark=df["close"],
)

sc = StrategyConnector(ps, toy_book, df["close"]) # instantiate StrategyConnector tool to analyse strategy integration into current toy book

sc.report()

In [ ]:
c = CapacityEstimator(Portfolio(df, cost_model=DynamicCostModel(), execution_model=CappedVolumeExecution()))
c.report()